## How to use this notebook

Run notebook 01 first, then run this one top to bottom. It builds `data/processed/alias_map.json`
and adds `canonical_ingredients` to `data/processed/cleaned_recipes.parquet`. The maintained alias
definitions live in `src/alias_map.py`; compound-aware matching lives in `src/ingredient_utils.py`.

# 02 - Filipino ingredient alias map
Input: `data/processed/cleaned_recipes.parquet` (from notebook 01). Needs `ingredient_utils.py` in the same folder as this notebook.

Outputs: `data/processed/alias_map.json` and an updated `cleaned_recipes.parquet` with `canonical_ingredients` and `unmapped_terms` columns.

**Workflow:** run the seed map, look at what it misses in your data, add entries, rerun. Repeat 2-4 times.

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Working directory:", Path.cwd())

## 1. Load

In [ ]:
import json, re
import pandas as pd
from collections import Counter
from rapidfuzz import process, fuzz
from ingredient_utils import line_to_phrases, build_lookup, match_canonicals, normalize_text
import ingredient_utils
print(ingredient_utils.__file__)
print("cored" in ingredient_utils.PREP, "stemmed" in ingredient_utils.PREP)

clean = pd.read_parquet("data/processed/cleaned_recipes.parquet")
print(clean.shape)

## 2. The alias map

Edit `src/alias_map.py` instead of this notebook. The cell below imports it, so there is one
definition shared by the notebooks, the tests, and the future RAG pipeline. Format:
`canonical name: [aliases]`. Only true synonyms go together; cuts and varieties stay separate,
and ambiguous words (e.g. `sili`) stay out until you decide.

In [ ]:
# Single source of truth: src/alias_map.py. Edit that file, not this notebook.
from alias_map import alias_map, parents_map, write_alias_map

ALIASES = alias_map()
PARENTS = parents_map()
print(len(ALIASES), "canonical ingredients,", len(PARENTS), "parent links")

## 3. Build the lookup and check for collisions
A collision means the same word is listed under two different canonical names. The list must be empty.

In [ ]:
lookup, collisions = build_lookup(ALIASES)
print(len(ALIASES), "canonical ingredients,", len(lookup), "terms in lookup")
print("collisions:", collisions if collisions else "none")

## 4. Apply to the data and measure coverage
Every raw ingredient line is cleaned into phrases, then matched against the map. Words that do not match anything are counted as *leftover*; the frequent ones are your to-do list.

In [ ]:
line_hits = Counter(); n_lines = n_matched = 0
leftover_words = Counter(); unmatched_phrases = Counter()
tok_total = tok_covered = 0

for lines in clean["ingredient_lines"]:
    for line in lines:
        n_lines += 1
        any_hit = False
        for ph in line_to_phrases(line):
            canon, left = match_canonicals(ph, lookup)
            ntok = len(ph.split())
            tok_total += ntok; tok_covered += ntok - len(left)
            if canon:
                any_hit = True
            else:
                unmatched_phrases[ph] += 1
            leftover_words.update(left)
        n_matched += any_hit

print(f"lines with at least one match: {n_matched}/{n_lines} ({n_matched/n_lines:.1%})")
print(f"tokens covered by the map:     {tok_covered}/{tok_total} ({tok_covered/tok_total:.1%})")

## 5. Your to-do list: what the map misses
**Left: whole phrases that matched nothing. Right: individual words that stayed uncovered.** Go through the top of both lists. For each term decide: is it an alias of something already in the map (add to its list), a new ingredient (add a new entry), or just noise like a cut or adjective (ignore).

In [ ]:
print("TOP UNMATCHED PHRASES")
for ph, n in unmatched_phrases.most_common(60):
    print(f"{n:4d}  {ph}")
print("\nTOP LEFTOVER WORDS")
for w, n in leftover_words.most_common(60):
    print(f"{n:4d}  {w}")

## 6. Spelling-variant suggestions
For frequent leftover words, find the closest existing term. Suggestions only; accept them by editing the map in section 2. Check each one, because close spelling does not always mean the same ingredient.

In [ ]:
known = list(lookup)
for w, n in leftover_words.most_common(80):
    best = process.extractOne(w, known, scorer=fuzz.ratio, score_cutoff=80)
    if best:
        print(f"{w!r:20} ({n}x)  ~  {best[0]!r} -> {lookup[best[0]]!r}  [{best[1]:.0f}]")

## 7. Sanity tests on known cases

In [ ]:
tests = {
    "toyo": "soy sauce", "patis": "fish sauce", "bawang": "garlic",
    "sitaw": "string beans", "snake beans": "string beans", "long green beans": "string beans",
    "Sibuyas": "onion", "4 cloves garlic, peeled and chopped": "garlic",
    "dahon ng laurel": "bay leaf", "kalamansi": "calamansi",
}
for text, want in tests.items():
    got, _ = normalize_text(text, lookup)
    print("OK  " if want in got else "FAIL", f"{text!r:42} -> {got}")

for q in ["toyo, bawang, sibuyas, manok", "baboy, sitaw, kamatis, patis"]:
    print("\nquery:", q)
    got = []
    for item in q.split(","):
        got += normalize_text(item, lookup)[0]
    print("  ->", got)

## 8. Save `alias_map.json`, and add the canonical columns to the recipes
Only run this when the coverage and to-do lists look good enough. It is fine to save an early version and come back later: just rerun sections 2 to 8.

In [ ]:
# Write the same map the pipeline will load; then annotate the recipes.
write_alias_map("data/processed/alias_map.json")

def row_canonicals(row):
    canon, left = [], []
    for text in list(row["ingredient_lines"]) + list(row["ingredient_list"]):
        c, l = normalize_text(text, lookup)
        canon += [x for x in c if x not in canon]
        left += l
    return canon, sorted(set(left))

res = clean.apply(row_canonicals, axis=1)
clean["canonical_ingredients"] = [r[0] for r in res]
clean["unmapped_terms"] = [r[1] for r in res]

print("recipes with zero canonical ingredients:", (clean["canonical_ingredients"].str.len() == 0).sum())
print("average canonical ingredients per recipe:", clean["canonical_ingredients"].str.len().mean().round(1))
clean.to_parquet("data/processed/cleaned_recipes.parquet", index=False)
clean[["recipe_name", "canonical_ingredients"]].sample(min(5, len(clean)), random_state=1)